In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ================================
# Step 1: Introduction & Setup
# ================================

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, KFold

# Gradient boosting libraries
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Kaggle file system (data directory)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Set plot style
sns.set(style="whitegrid", palette="muted", font_scale=1.2)

# Define RMSE evaluation helper
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


In [ ]:
# ================================
# Step 2: Load Data & Initial Exploration
# ================================

train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
sample_submission = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()


In [ ]:
# Basic info
train.info()

# Summary statistics
train.describe().T.head(10)

# Missing values count
missing_values = train.isnull().sum().sort_values(ascending=False)
missing_values[missing_values > 0].head(20)


In [ ]:
# ================================
# Step 3: Exploratory Data Analysis (EDA) & Visualization
# ================================

# Target variable distribution
plt.figure(figsize=(8,5))
sns.histplot(train['SalePrice'], kde=True, bins=40)
plt.title("SalePrice Distribution")
plt.show()

# Log-transform target (better for regression)
train["SalePrice_log"] = np.log1p(train["SalePrice"])

plt.figure(figsize=(8,5))
sns.histplot(train["SalePrice_log"], kde=True, bins=40, color="orange")
plt.title("Log-Transformed SalePrice Distribution")
plt.show()

# Correlation heatmap (numeric features)
numeric_feats = train.select_dtypes(include=[np.number])
corr = numeric_feats.corr()

plt.figure(figsize=(12,8))
sns.heatmap(corr['SalePrice'].sort_values(ascending=False).head(20).to_frame(),
            annot=True, cmap="coolwarm", cbar=False)
plt.title("Top Correlated Features with SalePrice")
plt.show()


In [ ]:
# ================================
# Step 4: Data Cleaning & Feature Engineering
# ================================

# Combine train and test for consistent preprocessing
train_ID = train['Id']
test_ID = test['Id']
y = train['SalePrice_log']  # use log target

train.drop(['Id','SalePrice','SalePrice_log'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

full_data = pd.concat([train, test], axis=0)

# Fill missing values
for col in full_data.columns:
    if full_data[col].dtype == "object":
        full_data[col] = full_data[col].fillna("None")
    else:
        full_data[col] = full_data[col].fillna(full_data[col].median())

# Encode categorical features
full_data = pd.get_dummies(full_data, drop_first=True)

# Split back into train/test
X = full_data.iloc[:len(y), :]
X_test_final = full_data.iloc[len(y):, :]

print("Processed Train Shape:", X.shape)
print("Processed Test Shape:", X_test_final.shape)


In [ ]:
# ================================
# Step 5: Model Training & Evaluation
# ================================

# Train-validation split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Dictionary to store model scores
model_scores = {}

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_valid)
model_scores['Linear Regression'] = rmse(y_valid, y_pred)

# 2. Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_valid)
model_scores['Decision Tree'] = rmse(y_valid, y_pred)

# 3. Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_valid)
model_scores['Random Forest'] = rmse(y_valid, y_pred)

# 4. Gradient Boosting
gbr = GradientBoostingRegressor(n_estimators=200, random_state=42)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_valid)
model_scores['Gradient Boosting'] = rmse(y_valid, y_pred)

# 5. XGBoost
xgbr = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
xgbr.fit(X_train, y_train)
y_pred = xgbr.predict(X_valid)
model_scores['XGBoost'] = rmse(y_valid, y_pred)

# 6. LightGBM
lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
lgbm.fit(X_train, y_train)
y_pred = lgbm.predict(X_valid)
model_scores['LightGBM'] = rmse(y_valid, y_pred)

# 7. CatBoost
catb = cb.CatBoostRegressor(iterations=500, learning_rate=0.05, verbose=0, random_state=42)
catb.fit(X_train, y_train)
y_pred = catb.predict(X_valid)
model_scores['CatBoost'] = rmse(y_valid, y_pred)

# 8. Neural Net
mlp = MLPRegressor(hidden_layer_sizes=(100,50), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_valid)
model_scores['Neural Net'] = rmse(y_valid, y_pred)


In [ ]:
# ================================
# Step 6: Model Comparison & Feature Importance
# ================================

# Create results table
results_df = pd.DataFrame(list(model_scores.items()), columns=["Model", "RMSE"]).sort_values("RMSE")
print(results_df)

# Plot comparison
plt.figure(figsize=(10,6))
sns.barplot(data=results_df, x="RMSE", y="Model", palette="viridis")
plt.title("Model RMSE Comparison")
plt.show()

# Feature importance (using best model)
best_model_name = results_df.iloc[0]["Model"]
print("Best Model:", best_model_name)

if best_model_name == "Random Forest":
    importances = rf.feature_importances_
elif best_model_name == "Gradient Boosting":
    importances = gbr.feature_importances_
elif best_model_name == "XGBoost":
    importances = xgbr.feature_importances_
elif best_model_name == "LightGBM":
    importances = lgbm.feature_importances_
elif best_model_name == "CatBoost":
    importances = catb.feature_importances_
else:
    importances = None

if importances is not None:
    feat_imp = pd.DataFrame({"Feature": X.columns, "Importance": importances})
    feat_imp = feat_imp.sort_values("Importance", ascending=False).head(10)

    plt.figure(figsize=(10,6))
    sns.barplot(data=feat_imp, x="Importance", y="Feature", palette="coolwarm")
    plt.title(f"Top 10 Features - {best_model_name}")
    plt.show()


In [ ]:
# ================================
# Step 7: Kaggle Submission
# ================================

# Retrain best model on all training data
if best_model_name == "Random Forest":
    best_model = rf
elif best_model_name == "Gradient Boosting":
    best_model = gbr
elif best_model_name == "XGBoost":
    best_model = xgbr
elif best_model_name == "LightGBM":
    best_model = lgbm
elif best_model_name == "CatBoost":
    best_model = catb
else:
    best_model = lr  # fallback

best_model.fit(X, y)
final_preds = best_model.predict(X_test_final)

# Convert back from log scale
final_preds_exp = np.expm1(final_preds)

submission = pd.DataFrame({"Id": test_ID, "SalePrice": final_preds_exp})
submission.to_csv("submission.csv", index=False)
print("Submission file created successfully!")
